# 06 - Human-in-the-loop com LangGraph

Caso prático: um **agente de análise de vendas**. Ele gera uma análise, mas não publica nada sozinho. O grafo para e espera uma pessoa aprovar. Se a pessoa rejeitar e deixar um feedback, o agente refaz a análise.

Fluxo:

```
START → gerar_analise → aprovacao ──(aprovar)──────────→ publicar → END
                            │
                            ├──(rejeitar, ainda há tentativas)──→ gerar_analise
                            └──(rejeitar, sem tentativas)───────→ encerrar → END
```

- **`interrupt(valor)`**: pausa o grafo no meio de um nó e envia `valor` para quem chamou. O grafo não avança até alguém retomar.
- **`Command(resume=...)`**: retoma o grafo. O valor vira o retorno de `interrupt(...)`.
- **Checkpointer (`MemorySaver`)**: guarda o estado entre a pausa e a retomada. Sem ele, `interrupt` não funciona.
- **`thread_id`**: identifica uma execução. Cada aprovação pendente fica presa a um `thread_id`.

## 1. Configuração

A chave e o modelo vêm do `.env` na raiz do projeto, como nos outros notebooks.

In [1]:
import os
from typing import TypedDict

from dotenv import find_dotenv, load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

load_dotenv(find_dotenv())

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

## 2. Estado

Além da análise, o estado guarda:

- `feedback`: o que a pessoa escreveu ao rejeitar. O próximo `gerar_analise` usa esse texto.
- `tentativas`: quantas análises já foram geradas. Serve de limite para o loop.
- `status`: em que ponto o fluxo está (`aprovada`, `rejeitada`, `publicada`, `encerrada sem aprovação`).

In [2]:
class Estado(TypedDict):
    tema: str
    dados: str
    analise: str
    feedback: str
    tentativas: int
    status: str


MAX_TENTATIVAS = 3

## 3. Nós

O nó `aprovacao` é o ponto de human-in-the-loop. Ele chama `interrupt` com o que a pessoa precisa ver. Quando o grafo for retomado, `interrupt` devolve a resposta dela, e o nó decide o próximo passo.

Um cuidado: ao retomar, o LangGraph **executa o nó de novo desde o início**. Por isso, nada antes do `interrupt` deve ter efeito colateral, como enviar e-mail ou gravar em banco. Efeitos desse tipo ficam depois da aprovação, no nó `publicar`.

In [3]:
def gerar_analise(estado: Estado) -> dict:
    instrucao = (
        "Você é um analista de vendas. Escreva uma análise em até 3 tópicos curtos, em português, "
        "baseada apenas nos dados fornecidos."
    )
    texto = f"Tema: {estado['tema']}\nDados: {estado['dados']}"
    if estado["feedback"]:
        texto += f"\nFeedback do revisor sobre a versão anterior: {estado['feedback']}"
    resposta = llm.invoke([SystemMessage(content=instrucao), HumanMessage(content=texto)])
    return {"analise": resposta.content, "tentativas": estado["tentativas"] + 1}


def aprovacao(estado: Estado) -> dict:
    resposta = interrupt({
        "analise": estado["analise"],
        "pergunta": "Aprovar esta análise? Responda 'aprovar' ou escreva o feedback para refazer.",
    })
    if resposta.strip().lower() == "aprovar":
        return {"status": "aprovada", "feedback": ""}
    return {"status": "rejeitada", "feedback": resposta}


def publicar(estado: Estado) -> dict:
    # Efeito externo: só roda depois da aprovação humana.
    print(f"[simulado] Análise publicada:\n{estado['analise']}")
    return {"status": "publicada"}


def encerrar(estado: Estado) -> dict:
    return {"status": "encerrada sem aprovação"}

## 4. Arestas condicionais e compilação

`apos_aprovacao` decide o caminho. Aprovada vai para `publicar`. Rejeitada volta para `gerar_analise`, a menos que o limite de tentativas tenha sido atingido.

`compile(checkpointer=MemorySaver())` ativa a pausa e a retomada.

In [4]:
def apos_aprovacao(estado: Estado) -> str:
    if estado["status"] == "aprovada":
        return "publicar"
    if estado["tentativas"] >= MAX_TENTATIVAS:
        return "encerrar"
    return "gerar_analise"


grafo = StateGraph(Estado)
grafo.add_node("gerar_analise", gerar_analise)
grafo.add_node("aprovacao", aprovacao)
grafo.add_node("publicar", publicar)
grafo.add_node("encerrar", encerrar)

grafo.add_edge(START, "gerar_analise")
grafo.add_edge("gerar_analise", "aprovacao")
grafo.add_conditional_edges("aprovacao", apos_aprovacao, ["publicar", "encerrar", "gerar_analise"])
grafo.add_edge("publicar", END)
grafo.add_edge("encerrar", END)

app = grafo.compile(checkpointer=MemorySaver())

## 5. Visualizando o grafo

O diagrama em Mermaid mostra as três saídas de `aprovacao`.

In [5]:
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	gerar_analise(gerar_analise)
	aprovacao(aprovacao)
	publicar(publicar)
	encerrar(encerrar)
	__end__([<p>__end__</p>]):::last
	__start__ --> gerar_analise;
	aprovacao -.-> encerrar;
	aprovacao -.-> gerar_analise;
	aprovacao -.-> publicar;
	gerar_analise --> aprovacao;
	encerrar --> __end__;
	publicar --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 6. Função auxiliar

`pausa_atual` lê o estado salvo para uma `thread_id` e mostra o que está esperando aprovação. Quem usa o grafo em produção faria isso numa tela ou num aplicativo de chat.

In [6]:
DADOS = "Vendas mensais de 2026 (R$ mil): jan 120, fev 98, mar 143, abr 110."


def pausa_atual(config: dict) -> None:
    estado = app.get_state(config)
    for tarefa in estado.tasks:
        for interrupcao in tarefa.interrupts:
            print("PAUSADO aguardando aprovação.\n")
            print("Análise:")
            print(interrupcao.value["analise"])
            print("\n" + interrupcao.value["pergunta"])

## 7. Primeira execução: o grafo para antes de publicar

`invoke` roda até a primeira `interrupt`. Nesse ponto ele retorna sem publicar. O estado fica salvo na `thread_id`.

In [7]:
config = {"configurable": {"thread_id": "relatorio-vendas"}}

app.invoke(
    {"tema": "Vendas do primeiro quadrimestre", "dados": DADOS, "feedback": "", "tentativas": 0},
    config,
)
pausa_atual(config)

PAUSADO aguardando aprovação.

Análise:
1) Resumo numérico: total do 1º quadrimestre = R$ 471 mil; média mensal = R$ 117,8 mil. Melhor mês: março (R$ 143 mil); pior mês: fevereiro (R$ 98 mil).

2) Variações mês a mês: jan→fev -18,3%; fev→mar +45,9%; mar→abr -23,1%. Resultado líquido jan→abr: -8,3% (120 → 110).

3) Interpretação curta: vendas voláteis com pico em março que elevou a média; necessidade de investigar causa da alta em mar/quedas em fev e abr para buscar estabilidade.

Aprovar esta análise? Responda 'aprovar' ou escreva o feedback para refazer.


## 8. Rejeitando com feedback

`Command(resume=...)` retoma a mesma `thread_id`. O texto enviado vira o retorno de `interrupt`. Como não é `aprovar`, o nó marca a análise como rejeitada, grava o feedback e o grafo volta a `gerar_analise`. A nova análise pausa de novo.

In [8]:
app.invoke(Command(resume="Compare com o mesmo período de 2025 e explique a queda de fevereiro."), config)
pausa_atual(config)

PAUSADO aguardando aprovação.

Análise:
- Resumo do 1.º quadrimestre: vendas totais R$ 471 mil; média mensal R$ 117,8 mil; pico em mar (R$ 143 mil) e menor valor em fev (R$ 98 mil).

- Dinâmica mensal: jan 120 → fev 98 (queda de 18,3%); fev → mar 98 → 143 (alta de 45,9%); mar → abr 143 → 110 (queda de 23,1%).

- Sobre comparar com 2025 e explicar a queda de fevereiro: não é possível fazer comparação nem atribuir causalidade apenas com os dados fornecidos (faltam os números de 2025 e variáveis operacionais). Para investigar, é preciso obter o 1.º quadrimestre de 2025 e informações sobre promoções, mix de produtos, canais, preços e eventos/saídas de clientes.

Aprovar esta análise? Responda 'aprovar' ou escreva o feedback para refazer.


## 9. Aprovando

Agora a resposta é `aprovar`. O grafo segue para `publicar` e termina.

In [9]:
resultado = app.invoke(Command(resume="aprovar"), config)
print("Status final:", resultado["status"])

[simulado] Análise publicada:
- Resumo do 1.º quadrimestre: vendas totais R$ 471 mil; média mensal R$ 117,8 mil; pico em mar (R$ 143 mil) e menor valor em fev (R$ 98 mil).

- Dinâmica mensal: jan 120 → fev 98 (queda de 18,3%); fev → mar 98 → 143 (alta de 45,9%); mar → abr 143 → 110 (queda de 23,1%).

- Sobre comparar com 2025 e explicar a queda de fevereiro: não é possível fazer comparação nem atribuir causalidade apenas com os dados fornecidos (faltam os números de 2025 e variáveis operacionais). Para investigar, é preciso obter o 1.º quadrimestre de 2025 e informações sobre promoções, mix de produtos, canais, preços e eventos/saídas de clientes.
Status final: publicada


## 10. Limite de tentativas

Se a pessoa rejeitar sempre, o loop precisa parar. Depois de `MAX_TENTATIVAS` análises, `apos_aprovacao` manda o fluxo para `encerrar`. Este bloco usa outra `thread_id` e rejeita três vezes.

In [10]:
config_limite = {"configurable": {"thread_id": "relatorio-limite"}}

app.invoke(
    {"tema": "Vendas do primeiro quadrimestre", "dados": DADOS, "feedback": "", "tentativas": 0},
    config_limite,
)
for _ in range(MAX_TENTATIVAS):
    resultado = app.invoke(Command(resume="Não está bom, refaça."), config_limite)

print("Tentativas:", resultado["tentativas"])
print("Status final:", resultado["status"])

Tentativas: 3
Status final: encerrada sem aprovação


## 11. Por que isso importa

- **Pausa sem processo parado**: o estado fica no checkpointer. Quem aprova pode responder minutos ou dias depois, em outra requisição.
- **Resposta tipada pelo código**: o nó compara `aprovar` com a resposta. Para casos de produção, convém validar a resposta antes de retomar.
- **Efeito externo só depois da aprovação**: `publicar` é o único nó com efeito externo, e ele roda depois da pausa.
- **Limite de loop**: sem `MAX_TENTATIVAS`, uma rejeição infinita geraria chamadas ao LLM para sempre.